# Kaggle runner — LLM reliability study
Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**, persistence: files.

Secrets (Add-ons → Secrets): `HF_TOKEN` (Hugging Face, Llama-3.2 licence accepted) and `GH_TOKEN` (GitHub fine-grained token, read-only on `krisy1/llm-reliability`).

Wordings must already be finalized locally and committed (`data/reworded/`). Generation is resumable: if the session dies, rerun from the generation cell. For long runs use **Save Version → Save & Run All** so it runs in the background (12 h limit).

In [ ]:
# 0. Secrets
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
GH = sec.get_secret('GH_TOKEN')

In [ ]:
# 1. Get the code (private repo)
%cd /kaggle/working
!git clone -q https://{GH}@github.com/krisy1/llm-reliability.git
%cd /kaggle/working/llm-reliability
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Optional: vLLM is usually much faster; if it fails, use the default transformers backend
!pip install -q vllm || echo 'vLLM not available -> use the transformers backend'

In [ ]:
# 3. Data, reshuffles, full manifest (300 items x 3 wordings x 2 orders x (4 seeds + greedy) = 9,000 runs per model)
!python -m src.sample_mmlu --n 300
!python -m src.make_variants orders --items data/sampled/mmlu_pro_n300.jsonl
!python -m src.build_manifest --items data/sampled/mmlu_pro_n300.jsonl --seeds 0 1 2 3 --wordings w0 w1 w2 --name mmlu300_full

In [ ]:
# 4. M1 checks: timing of the largest model on 50 items, and the prefill pilot (greedy, w0)
!python -m src.build_manifest --items data/sampled/mmlu_pro_n300.jsonl --seeds 0 1 2 3 --wordings w0 w1 w2 --limit 50 --name timing50
!python -m src.run_generation --model phi3.5-mini --manifest data/manifests/timing50.jsonl --batch-size 32
!tail -n 3 results/logs/timing.jsonl
!python -m src.build_manifest --items data/sampled/mmlu_pro_n300.jsonl --greedy-only --wordings w0 --limit 50 --name pilot_prefill_greedy
for m in ['qwen2.5-3b', 'llama3.2-3b']:
    !python -m src.run_generation --model {m} --manifest data/manifests/pilot_prefill_greedy.jsonl
    !python -m src.run_generation --model {m} --manifest data/manifests/pilot_prefill_greedy.jsonl --prefill "The answer is" --tag _prefill
!python -m src.score_runs --manifest data/manifests/pilot_prefill_greedy.jsonl
# Rule: use the prefill for ALL models only if a 3B model's parse-failure rate is above 10% without it.

In [ ]:
# 5. Full MMLU-Pro run, one model per cell (resumable). Add --backend vllm if it installed.
MODEL = 'qwen2.5-3b'   # then llama3.2-3b, phi3.5-mini, smollm3-3b, granite3.3-2b
!python -m src.run_generation --model {MODEL} --manifest data/manifests/mmlu300_full.jsonl --batch-size 32

In [ ]:
# 6. Score + analyse
!python -m src.score_runs --manifest data/manifests/mmlu300_full.jsonl
!python -m src.analyse --scored results/scored/mmlu300_full.csv --ranking-only

## BFCL multiple (function calling)
Same steps with `--config configs/experiment_bfcl.yaml`. Pilot on 100 items first (`--n 100`), then all 200.

In [ ]:
N = 100   # pilot; set to 200 for the full run
!python -m src.sample_bfcl --n {N}
!python -m src.make_variants orders --items data/sampled/bfcl_multiple_n{N}.jsonl --config configs/experiment_bfcl.yaml
!python -m src.build_manifest --config configs/experiment_bfcl.yaml --items data/sampled/bfcl_multiple_n{N}.jsonl --name bfcl{N}

In [ ]:
MODEL = 'qwen2.5-3b'
!python -m src.run_generation --config configs/experiment_bfcl.yaml --model {MODEL} --manifest data/manifests/bfcl{N}.jsonl --batch-size 16

In [ ]:
!python -m src.score_runs --manifest data/manifests/bfcl{N}.jsonl
!python -m src.analyse --config configs/experiment_bfcl.yaml --scored results/scored/bfcl{N}.csv --ranking-only

In [ ]:
# 7. Save outputs before the session ends (files in /kaggle/working are kept in the saved version's Output)
!zip -qr /kaggle/working/results.zip results data/manifests
!ls -lh /kaggle/working/results.zip